# 01 — Baseline Tabular (M2)
MLP + XGBoost/LightGBM cho `Plaque_present`, hồi quy `Baseline_Risk_Score`, baseline LDL-only.
> Chạy cell setup (giống `00_setup_colab.ipynb`) trước.

In [ ]:
# --- Setup (copy tu 00) ---
SOURCE = "git"
GIT_URL = "https://github.com/<user>/Master-UIT-MedSignal.git"
DRIVE_PATH = "/content/drive/MyDrive/Master-UIT-MedSignal"
import os, sys
if SOURCE == "drive":
    from google.colab import drive; drive.mount('/content/drive'); PROJECT = DRIVE_PATH
else:
    PROJECT = "/content/Master-UIT-MedSignal"
    if not os.path.exists(PROJECT): os.system(f"git clone {GIT_URL} {PROJECT}")
os.chdir(PROJECT); sys.path.append(PROJECT)
!pip install -q -r requirements.txt

In [ ]:
from src.data import preprocess as P
from src.data.splits import stratified_folds
from src.models.baselines import build_tree_classifier, build_risk_regressor, ldl_only_features
from src.eval.metrics import classification_metrics, aggregate_folds

cfg = P.load_config("configs/config.yaml")
df = P.encode_categorical(P.load_dataframe(cfg), cfg)
folds = stratified_folds(df, cfg)
feat_cols = P.feature_columns(cfg)
ycol = cfg['columns']['target_plaque']

In [ ]:
# M2 TODO: vong 5-fold XGBoost cho Plaque_present
fold_metrics = []
for tr, va in folds:
    scaler = P.fit_scaler(df.iloc[tr], cfg)
    Xtr = P.apply_scaler(df.iloc[tr], scaler, cfg)[feat_cols].values
    Xva = P.apply_scaler(df.iloc[va], scaler, cfg)[feat_cols].values
    ytr, yva = df.iloc[tr][ycol].values, df.iloc[va][ycol].values
    clf = build_tree_classifier('xgboost', scale_pos_weight=(ytr==0).sum()/(ytr==1).sum())
    clf.fit(Xtr, ytr)
    prob = clf.predict_proba(Xva)[:, 1]
    fold_metrics.append(classification_metrics(yva, prob))
print(aggregate_folds(fold_metrics))

**M2 TODO tiếp theo:** MLP `TabularClassifier`, hồi quy risk score (`build_risk_regressor`), baseline LDL-only & lipid-panel để đối chứng discordance, lưu kết quả vào bảng so sánh chung (Phần 4 PROJECT_PLAN).